# B2-024-gpu-scientific-ml-capstone — Practice p28 — Solution

**Type:** challenge · **Difficulty:** advanced · **Concepts:** open-ended-model-evaluation

*55 minutes.*  
**Set:** C  
**Program layer:** Round 2 extension  
**Compute:** `compute.policy: cpu` · seed `20261022`  
**Qualified prerequisites:** `book1:F1-scientific-python`, `book1:F3-matrices`, `book1:F4-multivar-calculus`, `book1:F5-probability`, `book1:C1-ml-fundamentals`, `book1:C2-linear-models`, `book1:C3-gradient-descent`, `book1:C5-neural-networks`, `book1:C6-pytorch`, `book1:C7-cnn-transfer`, `book1:C10-competition-craft`, `book1:C11-neural-training`, `book1:C12-classical-models`, `B2-023-generative-models-diffusion`  
**Remediation links actually used:** [book1:F1-scientific-python](../../../../book1/units/F1-scientific-python/lesson.ipynb), [book1:F3-matrices](../../../../book1/units/F3-matrices/lesson.ipynb), [book1:F4-multivar-calculus](../../../../book1/units/F4-multivar-calculus/lesson.ipynb), [book1:F5-probability](../../../../book1/units/F5-probability/lesson.ipynb), [book1:C1-ml-fundamentals](../../../../book1/units/C1-ml-fundamentals/lesson.ipynb), [book1:C2-linear-models](../../../../book1/units/C2-linear-models/lesson.ipynb), [book1:C3-gradient-descent](../../../../book1/units/C3-gradient-descent/lesson.ipynb), [book1:C5-neural-networks](../../../../book1/units/C5-neural-networks/lesson.ipynb), [book1:C6-pytorch](../../../../book1/units/C6-pytorch/lesson.ipynb), [book1:C7-cnn-transfer](../../../../book1/units/C7-cnn-transfer/lesson.ipynb), [book1:C10-competition-craft](../../../../book1/units/C10-competition-craft/lesson.ipynb), [book1:C11-neural-training](../../../../book1/units/C11-neural-training/lesson.ipynb), [book1:C12-classical-models](../../../../book1/units/C12-classical-models/lesson.ipynb), [B2-023-generative-models-diffusion](../../B2-023-generative-models-diffusion/lesson.ipynb).

## Supplied cell (copied verbatim from the statement)

In [ ]:
import math
import torch

SEED_SCORES = {0: 0.0226, 1: 0.0219, 2: 0.0231, 3: 0.0197, 4: 0.0224}   # MLP-128, 1500 steps
N_TEST = 200

## Solution

### (A) Test-set reuse

The locked test set was scored **nine times** (runs 1–9), and every decision in the report was made with those scores: the width (64 vs 128 vs 256: runs 1–2 and 3 vs 5), the step count (1,000 vs 1,500: runs 2 vs 3), dropout (runs 3 vs 4), and finally which seed to report (runs 3 and 6–9).
The reported $0.0197$ is the **minimum of nine noisy test evaluations** chosen *because* it was the smallest.
Even if every other defect were fixed, the minimum of several noisy estimates is biased downward (lower is better here), exactly like the best of several seeds: part of its advantage is the luck of those 200 examples and that seed, which a fresh test set would not reproduce. The test set has become a selection set, and no unbiased test number remains.

### (B) Seed cherry-picking

In [ ]:
scores = [SEED_SCORES[s] for s in sorted(SEED_SCORES)]
r = len(scores)
seed_mean = sum(scores) / r
seed_sd = math.sqrt(sum((v - seed_mean) ** 2 for v in scores) / (r - 1))      # sample sd, divisor r - 1
seed_sd_torch = torch.tensor(scores, dtype=torch.float64).std().item()
reported_seed, reported = 3, SEED_SCORES[3]
rank = 1 + sum(v < reported for v in scores)            # 1 = lowest (best) error
gap = reported - seed_mean
print(f"seed scores {scores}: mean {seed_mean:.5f}, sample sd {seed_sd:.6f} (torch.std {seed_sd_torch:.6f})")
print(f"reported seed {reported_seed} ({reported}) has rank {rank} of {r}; gap to the mean {gap:+.5f} ({gap / seed_sd:+.2f} sd)")

The reported configuration's five seeds average $0.02194$ with sample standard deviation $0.00132$. The reported seed 3 ($0.0197$) is **rank 1 of 5** (the lowest error), $0.00224$ below the mean, i.e. about $1.7$ seed standard deviations: it is the luckiest seed, and it is the only one shown in the claim (the other four are in the table but ignored in the headline). Seed 3 was also the seed on which every configuration was compared (runs 1–5), so the configuration choice was made on the same lucky seed.

### (C) Invalid interval

In [ ]:
half_width = 1.96 * seed_sd / math.sqrt(N_TEST)
recomputed = (reported - half_width, reported + half_width)
print(f"report's formula: half-width {half_width:.6f} -> [{recomputed[0]:.5f}, {recomputed[1]:.5f}] "
      f"(rounds to [{recomputed[0]:.4f}, {recomputed[1]:.4f}])")
# For contrast: a seed-level interval for the configuration's mean score (t quantile, 4 degrees of freedom)
t_4 = 2.776
seed_half = t_4 * seed_sd / math.sqrt(r)
print(f"seed-mean interval (still test-tuned): {seed_mean:.4f} +/- {seed_half:.4f} -> [{seed_mean - seed_half:.4f}, {seed_mean + seed_half:.4f}]")

Errors in the construction:

1. **Wrong $n$.** $\sqrt{200}$ is the number of *test examples*, but the standard deviation was computed over $r=5$ *seed scores*. A standard error of a mean of 5 seed scores divides by $\sqrt5$, not $\sqrt{200}$; dividing by $\sqrt{200}$ shrinks the interval by a factor $\sqrt{40}\approx6.3$.
2. **Wrong standard deviation for the claimed interval.** The sample sd of seed scores measures seed-to-seed (training) variability of the mean test error; an interval "over test examples" needs the variability of the metric over **examples**, i.e. the spread of the 200 per-example errors (or a bootstrap over them). The formula mixes one quantity's spread with the other's sample size.
3. **Wrong centre.** It is centred on the best run (the cherry-picked seed-3 score, itself chosen as the minimum of nine test scores) rather than on an honest estimate such as the seed mean; with five seeds, a seed-level interval would also need the $t_4$ quantile ($2.776$), not $1.96$.
4. It ignores the selection in (A) and the leakage in (D), which no formula on these numbers can repair.

Checking the arithmetic: $1.96\cdot0.001324/\sqrt{200}=0.000183$, giving $[0.01952,0.01988]$, which rounds to the reported $[0.0195,0.0199]$; so the arithmetic is right and the method is wrong. (For contrast, the seed-level interval for the configuration's mean is $0.0219\pm0.0016$, about nine times wider, and still test-tuned.)

A valid 95% interval over test examples requires the **per-example test errors** of one pre-chosen model (the 200 permutation-invariant errors), resampled with a percentile bootstrap (or a standard error from their spread) — exactly what `final_test_score` returns — computed in **a single test call** made after selection. The report contains only nine aggregate test means, so no valid example-level interval can be computed from it.

### (D) Leakage

The inputs were standardized with the mean and standard deviation of all 1,200 rows, so the 200 **validation** and 200 **test** functions' sampled values contributed to the per-feature mean and scale that every training input was transformed with; the model is therefore fitted in a coordinate system partly defined by the test inputs, and at prediction time the test rows are standardized with statistics that include themselves.
No test label was used, but held-out *input* information was: the preprocessing is a fitted quantity, and fitting it on test rows tunes the representation to the test distribution, so test performance is no longer a measurement on unseen data and is (typically) optimistically biased by an amount the report cannot quantify. The fix is to fit the statistics on the 800 training rows only (and keep them as buffers inside the model).

### (E) Corrected report

**Corrected protocol.** All nine runs must be **rerun**, because every one of them used preprocessing fitted on all 1,200 rows; the standardization statistics are recomputed from the 800 training rows only and stored inside the model. Each candidate configuration (MLP-64/1000, MLP-128/1000, MLP-128/1500, MLP-128/1500+dropout, MLP-256/1500) is trained with the **same** seeds (e.g. 0–4) and the same step budget per comparison, and scored by its mean **validation** error over seeds, with per-seed paired differences; the configuration, step count and dropout are chosen on validation by a rule written before the runs (e.g. lowest mean, preferring the smaller model unless the paired interval excludes 0). The locked test set is then scored **once**, for the chosen configuration's pre-designated model (e.g. seed 0, or the seed ensemble), with `final_test_score`, which returns the error and a bootstrap interval over the 200 test examples; the committed least-squares baseline's $0.0091$ (same rows, same metric, `baseline_score("mixture")`) is reported beside it.

**Corrected results paragraph.** The current numbers support only a weak, qualitative statement: on (leaky, test-selected) scores, wider and longer-trained MLPs were somewhat better than MLP-64/1000, and for MLP-128/1500 the seed-to-seed spread ($\hat\sigma\approx0.0013$ over five seeds, mean $0.0219$) is as large as the differences between configurations measured on seed 3. They cannot support the claim "$0.0197$ (95% CI $[0.0195,0.0199]$)": that number is the best of nine test evaluations, the interval mixes seed spread with the test-set size and is centred on a cherry-picked seed, and the preprocessing leaked held-out inputs. Every one of these MLPs is also far **worse** than the committed per-function least-squares baseline ($0.0091$), so even an honest rerun would have to report that the raw regressor does not beat the baseline, while a regressor-plus-refinement pipeline (practice p20) is the kind of method that can. Until the rerun, no test number from this report should be quoted.

### Answer check

In [ ]:
manual_mean = (0.0226 + 0.0219 + 0.0231 + 0.0197 + 0.0224) / 5
assert abs(seed_mean - manual_mean) < 1e-9 and abs(seed_mean - 0.02194) < 1e-9
manual_sd = math.sqrt(sum((v - manual_mean) ** 2 for v in scores) / 4)
assert abs(seed_sd - manual_sd) < 1e-9
assert abs(seed_sd - seed_sd_torch) < 1e-9
assert abs(seed_sd - math.sqrt(1.753e-6)) < 1e-9          # squared deviations sum to 7.012e-6
assert rank == 1
assert abs(gap - (-0.00224)) < 1e-9
assert abs(half_width - 1.96 * seed_sd / math.sqrt(200)) < 1e-15
assert abs(half_width - 0.0001835) < 1e-7
assert round(recomputed[0], 4) == 0.0195 and round(recomputed[1], 4) == 0.0199